SEQUENCE TO SEQUENCE (SEQ2SEQ): L'ARCHITETTURA CHE HA UNITO LE LINGUE

Sequence to Sequence è un'architettura pensata per trasformare una sequenza in un'altra anche quando input ed output hanno lunghezze diverse.

Finora abbiamo visto modelli che associano una sequenza a un'etichetta. Ma come facciamo quando l'output deve essere un'altra sequenza di lunghezza diversa, come nella traduzione dell'inglese all'italiano?
La soluzione è sdoppiare il sistema in due componenti: uno che legge e 'comprime' l'input (Encoder) e uno che 'decomprime' e genera l'output (Decoder)
* Encoder: (spesso LSTM o GRU) è come un lettore vorace che prende appunti. Legge la frase, parola per parola, ed ad ogni passo aggiorna un riassunto interno. Alla fine della lettura abbiamo lo stato finale. 
* Stato finale: l'ultimo stato nascosto dell'encoder, che funge da riassunto completo della frase in ingresso.
* Decoder: è il nostro scrittore. Riceve solo gli appunti dell'Encoder ed inizia a generare la traduzione parola dopo parola, guidato da marcatori speciali che dicono quando iniziare e quando fermarsi. E' una seconda RNN che viene inizializzata con lo stato dell'encoder e genera la sequenza target parola dopo parola
* Token di Controllo: marcatori speciali come '<start>' e '<end>' per guidare l'inizio e la fine della generazione

Ma come fa, questo scrittore, a sapere cosa ha scritto un momento prima?
Il Ciclo di Generazione
Il decoder genera un token alla volta; l'output generato al tempo t diventa input per il tempo t+1, creando una catena di dipendenze.
Durante l'addestramento, passiamo al decoder la parola corretta del dataset invece della sua predizione per accelerare la convergenza. Questo permette di correggere direttamente invece di deragliare sull'errore iniziale.
In fase di test, invece, il modello deve basarsi esclusivamente sulle proprie predizioni precedenti, poichè non conosce la traduzione reale.

Non stiamo cercando una sola parola, ma la probabilità dell'intera sequenza. La probabilità della traduzione finale dipende da ogni parola precedente e da un contesto fondamentale.
Ma cos'è il contesto?

Il Vettore di Pensiero (Thought Vector)
Il cuore del trasferimento semantico.
Come può un computer 'capire' una frase prima di tradurla? il Thought Vector è la rappresentazione numerica compressa di tutto il significato dell'input. 
E' uno spazzio vettoriale dove il significato di 'mela' ed 'apple' dovrebbe essere identico.
E' il punto di giunzione dove l'Encoder finisce il suo lavoro e il Decoder inizia, portando con sè la memoria di ciò che è stato letto.

- Il vettore di contesto è un opera di compressione estrema. Riduce una sequenza di lunghezza variabile in un vettore denso di dimensione fissa (es 256 o 512). 
- Indipendenza dalla lingua: idealmente, frasi con lo stesso significato in lingue diverse, dovrebbero produrre vettori simili. Il vettore è indipendente dalla lingua, deve catturare l'essenza dell'azione, il soggetto ed il tempo verbale. In Keras questo vettore diventa lo stato iniziale del Decoder.
- Inizializzazione del Decoder: il vettore di contesto diventa lo stato nascosto iniziale del decoder
- Embedding dello Stato: racchiude non solo le parole ma anche le relazioni sintattiche e semantiche apprese.

Lo stato dell'Encoder non è statico, ogni passo dell'encoder aggiorna lo stato nascosto aggiungendo informazioni; l'ultimo stato 'h_n' è quello che chiamiamo vettore di contesto.
Il vettore vive in uno spazio astratto dove la vicinanza geometrica indica somiglianza concettuale tra intere frasi. Frasi con significati simili, anche se scritte in lingue diverse, saranno vicine.
Grazie a layer come LSTM o GRU, questo vettore riesce a preservare informaizoni dall'inzio della frase sorgente fino alla fine.

Esempio:
Input: "I love this movie"
Output: "Amo questo film"

Qui non stai facendo testo -> una classe, come nel sentiment analysis, ma da una sequenza ne generi un'altra.
Ed è proprio questo il significato di Sequence to Sequence.
La struttura classica di Seq2Seq ha due componenti:
Intpu -> ENCODER ->  rappresentazione dell'input -> DECODER -> sequenza di output
L'encoder legge la frese di input, il decoder genera la frase di output
Nel Seq2Seq classico, Encode e Decoder erano spesso costruiti usando:
- RNN
- LSTM 
- GRU

Qui c'è una differenza importante rispetto al sentiment analysis, nel Seq2Seq alla fine non vuoi una sola risposta, ma vuoi produrre una sequenza.
Il Decoder genera l'output un token alla volta
Il Decoder non genera necessariamente l'intera frase in un colpo solo. Nella forma classica autoregressiva genera token 1 -> token 2 -> token 3 -> ...
e il token precedente contribuisce alla generazione del successivo
Il Decodere non dice: "ora traduco la parola inglese numero 3"
dice piuttosto: "dato ciò che l'Encoder ha ricavato dalla frase inglese e dato ciò che ho già generato in italiano, qual'è il prossimo token italiano più probabile?"
Ed è proprio per questo che esistono, separatamente, Encoder e Decoder.
L'Encoder elabora la frase nella lingua sorgente
Il Decoder genera secondo la grammatica della lingua targeet
Quindi l'italiano viene generato secondo l'ordine italiano e non secondo l'ordine inglese.

Normalmente hai due vocabolari, un vocabolario di input (inglese nel nostro esempio) ed un vocabolario di output (italiano). Il Decoder deve scegliere il prossimo token dal vocabolario italiano.
Quindi il Decoder genera un token alla volta, ma non significa che traduca un token inlese alla volta.
Sono due concetti diversi.
il Seq2Seq fa:
intera sequenza inglese -> Encoder -> informazione sull'input -> Decoder - token 1 - token 2 - token 3 - ...

Per questo, nell'output, vengono utilizzati token speciali
<START> <END>
esempio
<START> amo questo file <END>
Il decoder parte da Start e cerca di predire "amo", poi ricevo "amo" e cerca di predire "questo" e così via fino a quando non riceve <END>

Input ed Output possono avere lunghezze diverse, per questo è necessario Encoder e Decoder separati. L'Encoder costruire una rappresentazione della sequenza di input, e il Decoder genera una nuova sequenza.
Questo rende Seq2Seq adatto:
- traduzione
- riassunto
- domanda - riposta
- trasformazione testuale

Parliamo di limiti
I limiti del Collo di Bottiglia
Quando la memoria non basta pià
Nonostante la potenza del Seq2Seq, affidare l'intera comprensione di una frase a un singolo vettore di dimensione fissa crea un problema critico.
Immaginate di dover riassumere un intero libro in una sola pagina: inevitabilmente, molti dettagli cruciali andranno persi nel processo di compressione.
Nel Seq2Seq classico, costringiamo l'intera complessità di una frase attraverso un unico vettore di dimensione fissa.
E' un limite fisico
E queali sono le conseguenze tecniche?

Il Problema del Bottleneck (collo di bottiglia)
- Saturazione informativa: il vettore di contesto non può contenere un'infinità di informazioni senza degradare la qualità
- Perdita del gradiente: per frasi molto lunghe, le informazioni iniziali faticano ad arrivare integre nel vettore finale
- Difficoltà di Allineamento: il decoder non sa quale parte dell'input ha generato una specifica parola dell'output. E' come cerca di tradurre un testo avendo a disposizione solo un riassunto.
- Decadimento delle prestazioni: l'accuratezza della traduzione crolla drasticamente all'aumentare della lunghezza della frase.

Conseguenze Pratiche
- Traduzioni Allucinate: il modello inizia a inventare parole o a ripetere segmenti quando non riesce più a recuperare informazioni dal vettore di contesto
- Omissione di dettagli: aggettivi, nomi propri o date posti all'inizio di una frase lunga, vengono spesso ignorati dal decoder
- Necessità di Attention: questi limiti hanno portato alla nascita dei meccanismi di attenzione, che vedremo nelle prossime lezioni per superare il bottleneck

Relazione tra Lunghezza della frase e Errore
Esiste una correlazione inversa tra il numero di token dell'input e la capacità del modello di mantenere la coerenza semantica.
Il bottle neck forza una rappresentazione troppo sintetica, rendendo impossibile catturare le sfumature di testi complessi. La qualità crolla quando il rapport tra la lunghezza dell'input e la lunghezza del contesto diventa troppo elevato. Più il testo è complesso più abbiamo bisogno di spazio di pensiero ma il bottleneck ce lo nega.

In [1]:
"""
================================================================================
SEQUENCE-TO-SEQUENCE (Seq2Seq) CON DATASET REALE
================================================================================
Obiettivo: Tradurre frasi dall'Inglese all'Italiano (EN -> IT).
Architettura: Encoder-Decoder con LSTM Bidirezionale .
Dataset: Hugging Face 'opus_books'.

"""

import os

# --- CONFIGURAZIONE AMBIENTE ---
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import keras
from keras import layers
import torch

# Verifica diagnostica
print(f"--- DIAGNOSTICA AMBIENTE ---")
print(f"Backend Keras: {keras.backend.backend()}")
print(f"GPU Disponibile (PyTorch): {torch.cuda.is_available()}")

def scarica_e_prepara_dati(num_samples=5000):
    """
    Gestisce il reperimento dei dati e la loro trasformazione in numeri (vettorizzazione).
    """
    try:
        from datasets import load_dataset
        print(f"--- 1. CARICAMENTO DATI ---")
        print(f"Recupero {num_samples} frasi da Hugging Face...")
        
        # Scarichiamo il dataset specializzato in traduzioni LIBRI (coppia en-it)
        dataset = load_dataset("opus_books", "en-it", split=f"train[:{num_samples}]", trust_remote_code=True)
        
        # Estraggono le frasi e le normalizziamo in minuscolo
        input_texts = [ex["translation"]["en"].lower() for ex in dataset]
        
        # Per il Decoder, aggiungiamo dei "marcatori di controllo":
        # 'starttoken' -> Dice al decoder: "Comincia a tradurre ora!"
        # 'endtoken'   -> Dice al decoder: "Hai finito, fermati!"
        target_texts = [f"starttoken {ex['translation']['it'].lower()} endtoken" for ex in dataset]
        print(f"Caricamento completato: {len(input_texts)} esempi pronti.")
        
    except Exception as e:
        print(f"\n[ERRORE] Hugging Face non disponibile: {e}. Uso dati sintetici.")
        input_texts = ["i am happy", "it is hot"] * (num_samples // 2)
        target_texts = ["starttoken sono felice endtoken", "starttoken fa caldo endtoken"] * (num_samples // 2)

    # --- 2. VETTORIZZAZIONE (Trasformazione testo -> numeri) ---
    max_tokens = 10000     # Dimensione massima del vocabolario (le 10k parole più comuni)
    sequence_length = 20   # Lunghezza fissa per ogni frase (se più corta, aggiunge zeri; se più lunga, taglia)

    # Vettorizzatore per l'Inglese (Input)
    src_vec = layers.TextVectorization(
        max_tokens=max_tokens,
        output_mode="int", # Converte ogni parola in un numero intero unico
        output_sequence_length=sequence_length,
    )
    
    # Vettorizzatore per l'Italiano (Output)
    tgt_vec = layers.TextVectorization(
        max_tokens=max_tokens,
        output_mode="int",
        output_sequence_length=sequence_length + 1, # Un passo in più per gestire lo shift temporale
    )

    # 'adapt' legge i testi per creare il dizionario parole-numeri
    src_vec.adapt(input_texts)
    tgt_vec.adapt(target_texts)

    return src_vec, tgt_vec, input_texts, target_texts

def costruisci_modello_seq2seq(src_vec, tgt_vec, latent_dim=512):
    """
    COSTRUZIONE DELL'ARCHITETTURA ENCODER-DECODER (Seq2Seq).
    
    Questo modello è composto da due parti principali che lavorano in tandem:
    1. ENCODER: Legge la frase in inglese e la 'condensa' in un vettore di stato (pensiero).
    2. DECODER: Prende quel vettore e 'srotola' la traduzione parola per parola.
    """
    
    # Recuperiamo il numero di parole uniche (token) dai vettorizzatori.
    # Serve per dimensionare correttamente gli strati di ingresso (Embedding).
    num_src_tokens = src_vec.vocabulary_size()
    num_tgt_tokens = tgt_vec.vocabulary_size()

    # ==========================================================================
    # PARTE A: L'ENCODER (La Comprensione)
    # --------------------------------------------------------------------------
    
    # 1. INPUT: Definiamo l'entrata per le sequenze di numeri interi (ID delle parole inglesi).
    # 'shape=(None,)' significa che accettiamo frasi di qualsiasi lunghezza.
    encoder_inputs = layers.Input(shape=(None,), dtype="int64", name="input_inglese")
    
    # 2. EMBEDDING: Trasforma ogni ID numerico in un vettore matematico denso di dimensione 'latent_dim'.
    # mask_zero=True è CRUCIALE: dice al modello di ignorare i PAD (zeri) usati per pareggiare le lunghezze.
    # Questo permette alla rete di concentrarsi solo sulle parole reali.
    x = layers.Embedding(num_src_tokens, latent_dim, mask_zero=True)(encoder_inputs)
    
    # 3. REGOLARIZZAZIONE: Spegniamo il 20% dei neuroni casualmente per evitare l'overfitting.
    x = layers.Dropout(0.2)(x)
    
    # 4. LSTM BIDIREZIONALE: È il 'cuore' dell'Encoder.
    # Legge la frase in avanti e all'indietro contemporaneamente.
    # return_state=True: Ci servono gli STATI FINALI (h e c) non solo gli output.
    # f_h, f_c: Stati del passaggio in avanti (Forward).
    # b_h, b_c: Stati del passaggio all'indietro (Backward).
    encoder_lstm = layers.Bidirectional(layers.LSTM(latent_dim, return_state=True))
    _, f_h, f_c, b_h, b_c = encoder_lstm(x)
    
    # 5. CONCATENAZIONE: Uniamo i due mondi (avanti e indietro).
    # Poiché abbiamo usato una LSTM Bidirezionale, dobbiamo unire gli stati.
    # Se latent_dim era 512, ora avremo un vettore di 1024 (512+512).
    state_h = layers.Concatenate()([f_h, b_h]) # Stato nascosto (memoria a breve termine)
    state_c = layers.Concatenate()([f_c, b_c]) # Stato della cella (memoria a lungo termine)
    
    # Questi due vettori insieme formano l'istante finale di comprensione della frase inglese.
    encoder_states = [state_h, state_c] 

    # ==========================================================================
    # PARTE B: IL DECODER (La Generazione)
    # --------------------------------------------------------------------------
    
    # 1. INPUT: L'ingresso per la traduzione italiana prodotta finora.
    # Durante il training usiamo il 'Teacher Forcing': diamo in pasto la risposta corretta shiftata.
    decoder_inputs = layers.Input(shape=(None,), dtype="int64", name="input_italiano_parziale")
    
    # 2. EMBEDDING ITALIANO: Proietta le parole italiane nello spazio vettoriale.
    x = layers.Embedding(num_tgt_tokens, latent_dim, mask_zero=True)(decoder_inputs)
    x = layers.Dropout(0.2)(x)
    
    # 3. LSTM DEL DECODER: Deve avere dimensione doppia (latent_dim * 2) perché
    # deve ospitare gli stati concatenati che arrivano dall'encoder bidirezionale.
    # return_sequences=True: Vogliamo l'output per OGNI parola della sequenza.
    decoder_lstm = layers.LSTM(latent_dim * 2, return_sequences=True, return_state=True)
    
    # 4. COLLEGAMENTO: Qui avviene la magia. Inizializziamo la memoria del decoder
    # con gli stati (h, c) prodotti dall'encoder. È il passaggio del testimone.
    decoder_outputs, _, _ = decoder_lstm(x, initial_state=encoder_states)
    
    # 5. STRATO DENSO FINALE: Trasforma i vettori interni (1024) in una distribuzione di
    # probabilità su tutto il vocabolario italiano (es. 10.000 parole).
    # activation="softmax": La somma di tutte le probabilità sarà 1.0.
    decoder_dense = layers.Dense(num_tgt_tokens, activation="softmax")
    decoder_outputs = decoder_dense(decoder_outputs)

    # --------------------------------------------------------------------------
    # CREAZIONE DEI MODELLI FUNZIONALI (Training vs Inference)
    # --------------------------------------------------------------------------
    # Perché creiamo 3 modelli invece di uno solo?
    # - In Training, diamo la frase intera (molto veloce, calcolo parallelo).
    # - In Inferenzia (traduzione), dobbiamo andare una parola alla volta (più complesso).

    # 1. MODELLO DI TRAINING (End-to-End)
    # -----------------------------------
    # Questo è il modello "maestro". Viene usato solo durante .fit().
    # Prende due ingressi: la frase inglese e la frase italiana (Teacher Forcing).
    # L'output è la previsione di tutta la traduzione slittata di un passo.
    model = keras.Model(
        inputs=[encoder_inputs, decoder_inputs], # Due entrate: sorgente e target parziale
        outputs=decoder_outputs,                # Una uscita: la probabilità delle parole italiane
        name="modello_training"
    )
    
    # 2. MODELLO ENCODER (Estrattore di Significato)
    # ----------------------------------------------
    # In fase di produzione (traduzione di una nuova frase), ci serve isolare l'Encoder.
    # Questo modello prende 'encoder_inputs' (la frase inglese) e restituisce
    # 'encoder_states' (il famoso "Thought Vector" con gli stati H e C della LSTM).
    # Non ci serve l'output della LSTM qui, ma solo la sua "memoria finale".
    encoder_model = keras.Model(
        inputs=encoder_inputs, 
        outputs=encoder_states, 
        name="encoder_solo"
    )
    
    # 3. MODELLO DECODER (Generatore Iterativo)
    # -----------------------------------------
    # Questa è la parte più complessa. Nella traduzione reale, il decoder non riceve
    # gli stati dall'encoder una sola volta, ma deve aggiornare la propria memoria
    # a ogni parola generata. Quindi creiamo un modello che "vive un passo alla volta".

    # Definiamo due nuovi Input per ricevere gli stati (H e C) "dall'esterno" 
    # (ovvero dal loop di traduzione che vedremo dopo).
    # La dimensione è latent_dim * 2 perché l'encoder era Bidirezionale.
    dec_state_h = layers.Input(shape=(latent_dim * 2,), name="stato_h_manuale")
    dec_state_c = layers.Input(shape=(latent_dim * 2,), name="stato_c_manuale")
    dec_states_inputs = [dec_state_h, dec_state_c]
    
    # Riapplichiamo gli strati già creati sopra (Embedding, LSTM, Dense).
    # È FONDAMENTALE riutilizzare gli stessi oggetti layer (decoder_lstm, decoder_dense)
    # così che i pesi imparati nel training siano gli stessi usati qui.
    
    # Passiamo la parola corrente attraverso l'embedding del decoder.
    dec_x = layers.Embedding(num_tgt_tokens, latent_dim, mask_zero=True)(decoder_inputs)
    
    # Chiamiamo la LSTM del decoder, ma stavolta passiamo 'initial_state' manualmente
    # usando gli input che abbiamo appena definito (dec_states_inputs).
    # Riceviamo in output: la previsione (dec_out) e i NUOVI stati aggiornati (s_h, s_c).
    dec_out, s_h, s_c = decoder_lstm(dec_x, initial_state=dec_states_inputs)
    
    # Trasformiamo l'uscita della LSTM in probabilità di parole reali.
    dec_out = decoder_dense(dec_out)
    
    # Infine, assembliamo il modello decoder per l'inferenza:
    # INPUT: [Parola attuale] + [Stati H e C precedenti]
    # OUTPUT: [Previsione parola successiva] + [Nuovi stati H e C]
    decoder_model = keras.Model(
        inputs=[decoder_inputs] + dec_states_inputs, 
        outputs=[dec_out, s_h, s_c], 
        name="decoder_solo"
    )

    return model, encoder_model, decoder_model

def translate(sentence, encoder, decoder, src_vec, tgt_vec):
    """
    LOGICA DI INFERENZA (TRADUZIONE):
    Poiché il modello lavora con sequenze, non possiamo tradurre tutto in un colpo solo.
    Dobbiamo generare una parola, aggiungerla alla frase, e usarla per generare la successiva.
    """
    
    # 1. ENCODING: Passiamo la frase inglese nell'Encoder.
    # Otteniamo il 'Thought Vector' (gli stati h e c) che rappresentano il significato.
    states = encoder.predict(src_vec([sentence]), verbose=0)
    
    # 2. DIZIONARIO: Prepariamo lo strumento per trasformare i numeri in parole leggibili.
    target_vocab = tgt_vec.get_vocabulary()
    lookup = dict(zip(range(len(target_vocab)), target_vocab))
    
    # 3. INIZIALIZZAZIONE: La prima parola passata al decoder è sempre 'starttoken'.
    # Creiamo una sequenza vuota che contiene solo il token di inizio.
    token_seq = np.zeros((1, 1))
    token_seq[0, 0] = target_vocab.index("starttoken")
    
    decoded_sentence = ""
    
    # 4. LOOP DI GENERAZIONE: Generiamo una parola alla volta (max 20 parole).
    for _ in range(20):
        # Chiediamo al decoder: "Dato questo pensiero e l'ultima parola scritta, cosa viene dopo?"
        output_tokens, h, c = decoder.predict([token_seq] + [states[0], states[1]], verbose=0)
        
        # Scegliamo la parola con la probabilità più alta (Argmax).
        sampled_index = np.argmax(output_tokens[0, -1, :])
        word = lookup[sampled_index]
        
        # Se la parola generata è 'endtoken', il modello ha deciso che la frase è finita.
        if word == "endtoken":
            break
            
        # Aggiungiamo la parola alla frase finale.
        decoded_sentence += " " + word
        
        # AGGIORNAMENTO STATO: L'output di questa iterazione diventa l'input della prossima.
        # Passiamo la parola appena generata e i nuovi stati h e c (la "memoria" aggiornata).
        token_seq[0, 0] = sampled_index
        states = [h, c]
        
    return decoded_sentence.strip()

# ==============================================================================
# BLOCCO DI ESECUZIONE (MAIN)
# ==============================================================================
if __name__ == "__main__":
    # 1. PREPARAZIONE DATI: Scarichiamo 5000 coppie di frasi.
    src_v, tgt_v, raw_in, raw_tgt = scarica_e_prepara_dati(5000)
    
    # 2. CREAZIONE ARCHITETTURA: Istanziamo i 3 modelli (training, encoder, decoder).
    model, encoder_m, decoder_m = costruisci_modello_seq2seq(src_v, tgt_v)

    # 3. LOGICA 'TEACHER FORCING' PER IL TRAINING:
    # Il decoder durante il training non impara da solo parola per parola.
    # Gli diamo la frase italiana intera (enc_in) ma "slittata".
    
    # encoder_in: La frase inglese originale.
    enc_in = src_v(raw_in)
    
    # decoder_in: La frase italiana SENZA l'ultima parola (inizia con starttoken).
    # Serve come "guida" per il decoder.
    dec_in = tgt_v([t.rsplit(' ', 1)[0] for t in raw_tgt])
    
    # decoder_tgt: La frase italiana SENZA la prima parola (finisce con endtoken).
    # È quello che il decoder deve imparare a predire per ogni parola di dec_in.
    dec_tgt = tgt_v([t.split(' ', 1)[1] for t in raw_tgt])

    # 4. COMPILAZIONE: Usiamo Adam per l'ottimizzazione e Sparse Categorical Crossentropy.
    # 'Sparse' perché i nostri target sono numeri interi (ID parole) e non vettori One-Hot.
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    
    # 5. TRAINING (IL MOMENTO DELLA VERITÀ):
    print("\n--- 3. ADDESTRAMENTO ---")
    print("Il modello imparerà a mappare il 'Pensiero' inglese sulla grammatica italiana.")
    # Passiamo [enc_in, dec_in] come input e dec_tgt come target.
    model.fit([enc_in, dec_in], dec_tgt, batch_size=64, epochs=50)

    # 6. TEST DI TRADUZIONE: Verifichiamo se ha imparato qualcosa.
    print("\n--- 4. TEST DI TRADUZIONE ---")
    test_phrases = ["i am happy", "the cat is black", "we love music"]
    for p in test_phrases:
        # Usiamo la funzione translate che usa i modelli di inference.
        traduzione = translate(p, encoder_m, decoder_m, src_v, tgt_v)
        print(f"INGLESE: '{p}'")
        print(f"ITALIANO: '{traduzione}'\n")

--- DIAGNOSTICA AMBIENTE ---
Backend Keras: torch
GPU Disponibile (PyTorch): True


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'opus_books' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


--- 1. CARICAMENTO DATI ---
Recupero 5000 frasi da Hugging Face...


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\barbara\.cache\huggingface\hub\datasets--opus_books. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



[ERRORE] Hugging Face non disponibile: Invalid HF URI 'hf://datasets/opus_books@1f9f6191d0e91a3c539c2595e2fe48fc1420de9b/.huggingface.yaml'. Repository id must be 'namespace/name', got 'opus_books'.. Uso dati sintetici.

--- 3. ADDESTRAMENTO ---
Il modello imparerà a mappare il 'Pensiero' inglese sulla grammatica italiana.
Epoch 1/50


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\keras\src\backend\torch\rnn.py:656: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  outputs, h_n, c_n = torch._VF.lstm(


79/79 ━━━━━━━━━━━━━━━━━━━━ 15s 168ms/step - accuracy: 0.9443 - loss: 0.1722
Epoch 2/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 170ms/step - accuracy: 1.0000 - loss: 4.6041e-06
Epoch 3/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 14s 178ms/step - accuracy: 1.0000 - loss: 3.0799e-06
Epoch 4/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 168ms/step - accuracy: 1.0000 - loss: 2.5162e-06
Epoch 5/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 12s 150ms/step - accuracy: 1.0000 - loss: 2.1447e-06
Epoch 6/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 169ms/step - accuracy: 1.0000 - loss: 1.8552e-06
Epoch 7/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 167ms/step - accuracy: 1.0000 - loss: 1.6278e-06
Epoch 8/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 14s 173ms/step - accuracy: 1.0000 - loss: 1.4402e-06
Epoch 9/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 14s 177ms/step - accuracy: 1.0000 - loss: 1.2839e-06
Epoch 10/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 162ms/step - accuracy: 1.0000 - loss: 1.1525e-06
Epoch 11/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 14s 173ms/step - accuracy: 1.0000 - loss: 1.0407e-06
Epoch 12/50
